# Exercise 2: Participial arc length

*Syntax and Universal Dependencies · version 0.1.0*

> First inspect how the treebank you choose labels relative clauses: some Turkish treebanks use plain `acl` for participial modifiers, so an empty `acl:relcl` query does not mean there are no relative clauses. Using TR-BOUN gold data (and, separately, the TurkicNLP Turkish pipeline), compute for each participial modifier the three quantities of the chapter's syntax sketch — clause width, the longest dependent-to-participle distance, and the participle-to-noun distance. Compare the participle-to-noun distance with the distance of `acl:relcl` arcs in English UD (EN-EWT). Plot the distributions and report the median and 90th percentile of each. Which of the three quantities differs most between the languages?

**Setup.** `pip install "turkicnlp==0.5.0" "stanza==1.11.0" scipy matplotlib requests`. Stanza is pinned to 1.11.0 because the worked solutions use its Turkish (IMST), Kazakh (KTB) and Kyrgyz (KTMU) checkpoints; with another version, Stanza downloads that version's models and the numbers can change. Models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`); missing ones are downloaded. The Universal Dependencies files are the **UD 2.17** release files (tag `r2.17` on GitHub). They are downloaded once into `UD217_DIR` (default `~/.cache/ud217`) and checked against the SHA-256 hashes recorded in the book's treebank inventory. Parsers receive the **gold words** (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores.

**Definitions.** A participial modifier is a word with deprel `acl` and `VerbForm=Part` (BOUN has no `acl:relcl`; the first cell prints the label inventory). *Clause width* is the number of non-punctuation words in the participle's subtree, the participle included. The *dependent-to-participle* distance is the largest distance from the participle to one of its own non-punctuation dependents (0 if it has none). The *participle-to-noun* distance is the distance to its head. For English the same quantities are computed for `acl:relcl` arcs. Quantiles use nearest rank, as in the chapter's arc-length table. The last part of the gold cell measures the pre-nominal material of nouns that head a participial clause, by the genre prefix of the BOUN sentence IDs (biographical, essays, instructional, news, popular culture).

In [ ]:
import collections, hashlib, math, os, random, statistics
from pathlib import Path
import requests

# Pinned UD 2.17 release files (the inventory in tables/tbl-ud217-inventory) and their SHA-256 hashes.
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {
    "UD_Turkish-BOUN/tr_boun-ud-train.conllu": "373fe39e3acdba3219521f54fa5ae768851fbb4bf9a31079ad1ca6590410650c",
    "UD_Turkish-BOUN/tr_boun-ud-test.conllu": "4f74fc0f9efdd1285d781f3efd1e98cbf39cf4433df9eb4dc14162e8ed36773b",
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Turkish-TueCL/tr_tuecl-ud-test.conllu": "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu": "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu": "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
    "UD_English-EWT/en_ewt-ud-train.conllu": "6fe5692f2e02198262483daba0626efe346dd19f0ce6c3ee7728b08af9b90a9c",
}
CACHE = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()

def ud_path(key):
    """Local copy of a pinned UD 2.17 file, downloaded once and checked against its SHA-256 hash."""
    path = CACHE / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos xpos feats head deprel")

def read(key_or_path):
    """Sentences as dicts: sent_id, text and words (integer-ID syntactic words only;
    multiword-token ranges and empty nodes are skipped)."""
    path = ud_path(key_or_path) if key_or_path in SHA256 else Path(key_or_path)
    sents, meta, words = [], {}, []
    for line in path.read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append({"sent_id": meta.get("sent_id"), "text": meta.get("text"), "words": words})
            meta, words = {}, []
        elif line.startswith("#"):
            k, _, v = line[1:].partition("=")
            meta.setdefault(k.strip(), v.strip())
        else:
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[4], c[5], int(c[6]), c[7]))
    return sents

def feats(w):
    return dict(f.split("=", 1) for f in w.feats.split("|") if "=" in f)

def children(words):
    out = collections.defaultdict(list)
    for w in words:
        out[w.head].append(w)
    return out

def subtree(words, i, kids=None):
    """IDs of word i and all its descendants."""
    kids = kids or children(words)
    out, stack = set(), [i]
    while stack:
        j = stack.pop(); out.add(j); stack.extend(k.id for k in kids[j])
    return out

def q(values, p):
    """Nearest-rank quantile, as in the chapter's arc-length table."""
    v = sorted(values)
    return v[math.ceil(len(v) * p) - 1] if v else None

def stanza_pipeline(lang, processors="tokenize,pos,lemma,depparse"):
    """A Stanza pipeline over gold words (tokenize_pretokenized=True). Uses models already on disk
    (STANZA_RESOURCES_DIR or ~/stanza_resources) and downloads them only if they are missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "depparse").exists()
    return stanza.Pipeline(lang, processors=processors, tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def parse_gold_words(nlp, sents, forms=None):
    """Parse each gold sentence from its gold words (optionally replaced by `forms`, one list per
    sentence, e.g. transliterated) and return predicted sentences with the original word forms."""
    forms = forms or [[w.form for w in s["words"]] for s in sents]
    doc = nlp(forms)
    out = []
    for s, ps in zip(sents, doc.sentences):
        assert len(ps.words) == len(s["words"]), s["sent_id"]
        out.append({"sent_id": s["sent_id"], "text": s["text"], "words": [
            Word(g.id, g.form, p.lemma or "_", p.upos or "_", p.xpos or "_", p.feats or "_", p.head, p.deprel)
            for g, p in zip(s["words"], ps.words)]})
    return out

def write_conllu(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(f"# sent_id = {s['sent_id']}\n# text = {s['text']}\n")
            for w in s["words"]:
                f.write("\t".join([str(w.id), w.form, w.lemma or "_", w.upos, w.xpos or "_", w.feats or "_",
                                   str(w.head), w.deprel, "_", "_"]) + "\n")
            f.write("\n")

def score(gold, pred):
    """UPOS, UAS and LAS (%) with the official CoNLL 2018 scorer, over gold-word input.
    Gold multiword tokens are not written, so both files have the same words and text."""
    import tempfile
    from stanza.utils import conll18_ud_eval as ev
    with tempfile.TemporaryDirectory() as tmp:
        g, p = Path(tmp) / "gold.conllu", Path(tmp) / "pred.conllu"
        write_conllu(gold, g); write_conllu(pred, p)
        r = ev.evaluate(ev.load_conllu_file(str(g)), ev.load_conllu_file(str(p)))
    return {k: round(100 * r[k].f1, 2) for k in ("Words", "UPOS", "UAS", "LAS")}

In [ ]:
def relative_clauses(sents, rel, verbform=None):
    """For each modifier arc with deprel `rel` (and, optionally, VerbForm on the modifier): the clause
    width (words in the modifier's subtree, punctuation excluded), the longest distance from the
    modifier to one of its own non-punctuation dependents (0 if it has none), and the
    modifier-to-noun distance."""
    rows = []
    for s in sents:
        ws = s["words"]; kids = children(ws); by_id = {w.id: w for w in ws}
        for w in ws:
            if w.deprel != rel or w.head == 0:
                continue
            if verbform and feats(w).get("VerbForm") != verbform:
                continue
            span = [j for j in subtree(ws, w.id, kids) if by_id[j].upos != "PUNCT"]
            dep = [abs(k.id - w.id) for k in kids[w.id] if k.upos != "PUNCT"]
            rows.append((len(span), max(dep, default=0), abs(w.id - w.head), w.id < w.head))
    return rows

def summary(label, rows):
    width, dep, noun, before = zip(*rows)
    print(f"{label}: {len(rows)} clauses; modifier before noun {sum(before) / len(rows):.1%}")
    for name, v in [("clause width", width), ("dependent-to-participle", dep), ("participle-to-noun", noun)]:
        print(f"   {name:24} median {q(v, .5):>2}  p90 {q(v, .9):>2}  mean {statistics.mean(v):.2f}")
    return width, dep, noun

boun = read("UD_Turkish-BOUN/tr_boun-ud-train.conllu")
print("BOUN acl subtypes:", dict(collections.Counter(w.deprel for s in boun for w in s["words"] if w.deprel.startswith("acl"))))
print("BOUN acl by VerbForm:", dict(collections.Counter(feats(w).get("VerbForm", "-") for s in boun for w in s["words"] if w.deprel == "acl")))
tr = summary("BOUN train, acl with VerbForm=Part", relative_clauses(boun, "acl", "Part"))
ewt = read("UD_English-EWT/en_ewt-ud-train.conllu")
en = summary("EWT train, acl:relcl", relative_clauses(ewt, "acl:relcl"))

# Pre-nominal material by genre: for every noun that heads at least one participial acl,
# the number of non-punctuation words of its subtree that precede it.
def prenominal(sents):
    by = collections.defaultdict(list)
    for s in sents:
        ws = s["words"]; kids = children(ws); by_id = {w.id: w for w in ws}
        genre = s["sent_id"].split("_")[0]
        for w in ws:
            if any(k.deprel == "acl" and feats(k).get("VerbForm") == "Part" for k in kids[w.id]):
                by[genre].append(sum(1 for j in subtree(ws, w.id, kids) if j < w.id and by_id[j].upos != "PUNCT"))
    return by
print("BOUN train, words before a noun that heads a participial clause, by sent_id genre prefix:")
for g, v in sorted(prenominal(boun).items(), key=lambda x: -len(x[1])):
    print(f"   {g:6} n={len(v):4}  median {q(v, .5)}  p90 {q(v, .9)}  max {max(v)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, name, a, b in zip(axes, ["clause width", "dependent-to-participle", "participle-to-noun"], tr, en):
    bins = range(0, 21)
    ax.hist([min(x, 20) for x in a], bins=bins, density=True, alpha=.6, label="Turkish BOUN (acl, Part)")
    ax.hist([min(x, 20) for x in b], bins=bins, density=True, alpha=.6, label="English EWT (acl:relcl)")
    ax.set_title(name); ax.set_xlabel("words (20 = 20 or more)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**The parser's view.** The Stanza Turkish parser (IMST models; TurkicNLP's default Turkish backend) parses the BOUN test sentences from gold words, and the same quantities are computed from its trees. The last part breaks attachment accuracy down by gold arc distance.

In [ ]:
# The parser's view: Stanza (Turkish IMST models) on the BOUN test sentences, from gold words.
import time
t0 = time.time()
nlp_tr = stanza_pipeline("tr")
boun_test = read("UD_Turkish-BOUN/tr_boun-ud-test.conllu")
pred = parse_gold_words(nlp_tr, boun_test)
print(f"parsed {len(pred)} sentences in {time.time() - t0:.0f} s; scores vs BOUN gold:", score(boun_test, pred))
summary("BOUN test, gold, acl with VerbForm=Part", relative_clauses(boun_test, "acl", "Part"))
# Stanza writes the IMST feature conventions; IMST also marks participles with VerbForm=Part.
summary("BOUN test, Stanza, acl with VerbForm=Part", relative_clauses(pred, "acl", "Part"))

def by_distance(gold, pred, bins=((1, 1), (2, 2), (3, 4), (5, 7), (8, 999))):
    """Head (UAS) and head+label (LAS) accuracy by gold arc distance; root and PUNCT excluded."""
    acc = {b: [0, 0, 0] for b in bins}
    for g, p in zip(gold, pred):
        for gw, pw in zip(g["words"], p["words"]):
            if gw.head == 0 or gw.upos == "PUNCT":
                continue
            d = abs(gw.id - gw.head)
            b = next(b for b in bins if b[0] <= d <= b[1])
            acc[b][0] += 1; acc[b][1] += pw.head == gw.head
            acc[b][2] += pw.head == gw.head and pw.deprel.split(":")[0] == gw.deprel.split(":")[0]
    for (lo, hi), (n, u, l) in acc.items():
        print(f"   gold distance {lo if lo == hi else f'{lo}-{hi}' if hi < 999 else f'{lo}+':>5}: {n:5} arcs  UAS {u / n:.1%}  LAS {l / n:.1%}")
print("Stanza on BOUN test, accuracy by gold arc distance:")
by_distance(boun_test, pred)

## Worked solution

Output of `experiments/ch08_exercises/run_ch08_exercises.py` (UD 2.17; Stanza 1.11.0):

```
BOUN acl subtypes: {'acl': 2791}
BOUN acl by VerbForm: {'-': 544, 'Part': 2233, 'Vnoun': 8, 'Conv': 6}
BOUN train, acl with VerbForm=Part: 2233 clauses; modifier before noun 99.7%
   clause width             median  4  p90 10  mean 4.91
   dependent-to-participle  median  1  p90  5  mean 2.35
   participle-to-noun       median  1  p90  4  mean 2.17
EWT train, acl:relcl: 1922 clauses; modifier before noun 0.0%
   clause width             median  6  p90 16  mean 8.21
   dependent-to-participle  median  3  p90  7  mean 3.77
   participle-to-noun       median  3  p90  8  mean 4.19
BOUN train, words before a noun that heads a participial clause, by sent_id genre prefix:
   news   n= 700  median 6  p90 13  max 46
   ess    n= 551  median 5  p90 13  max 75
   bio    n= 348  median 4  p90 10  max 31
   pop    n= 331  median 4  p90 11  max 32
   ins    n= 292  median 4  p90 11  max 40
```

```
parsed 979 sentences in 73 s; scores vs BOUN gold: {'Words': 100.0, 'UPOS': 87.79, 'UAS': 72.79, 'LAS': 63.17}
BOUN test, gold, acl with VerbForm=Part: 284 clauses; modifier before noun 100.0%
   clause width             median  3  p90  8  mean 4.56
   dependent-to-participle  median  1  p90  5  mean 2.12
   participle-to-noun       median  1  p90  4  mean 2.15
BOUN test, Stanza, acl with VerbForm=Part: 309 clauses; modifier before noun 99.7%
   clause width             median  4  p90  9  mean 4.90
   dependent-to-participle  median  1  p90  5  mean 2.16
   participle-to-noun       median  1  p90  3  mean 2.06
Stanza on BOUN test, accuracy by gold arc distance:
   gold distance     1:  5115 arcs  UAS 83.3%  LAS 69.4%
   gold distance     2:  1503 arcs  UAS 67.7%  LAS 54.0%
   gold distance   3-4:  1188 arcs  UAS 59.2%  LAS 48.1%
   gold distance   5-7:   699 arcs  UAS 54.2%  LAS 42.8%
   gold distance    8+:   700 arcs  UAS 43.9%  LAS 37.4%
```

**Labels first.** BOUN uses only plain `acl` (2,791 arcs in the training split), so an `acl:relcl` query would return nothing. 2,233 of these modifiers are participles (`VerbForm=Part`); 544 have no VerbForm, and 14 are converbs or verbal nouns.

**The three quantities.** For BOUN participles the medians are 4 words (clause width), 1 (dependent-to-participle) and 1 (participle-to-noun), with 90th percentiles of 10, 5 and 4. For English `acl:relcl` they are 6, 3 and 3, with 90th percentiles of 16, 7 and 8. The participle-to-noun distance differs most in relative terms: the Turkish participle is normally adjacent to its noun (median 1), whereas the English relative verb follows the relative pronoun and the clause's subject (median 3, 90th percentile 8). Turkish relative clauses are also narrower than English ones in these two corpora, so the stereotype of very long Turkish relative clauses describes the tail, not the typical case. Direction is categorical: 99.7% of Turkish participles precede their noun, no English relative clause does.

**Pre-nominal material by genre.** For nouns that head a participial clause, the median number of words before the noun is 6 in news, 5 in essays and 4 in the other genres, with 90th percentiles of 10–13 words. The maximum is 75 words, in an essay.

**The parser.** Parsing the BOUN test set with the IMST-trained Stanza model gives LAS 63.17 (UAS 72.79) from gold words, a cross-treebank score. It finds 309 participial `acl` arcs where the gold trees have 284, but the distance distributions are almost the same as gold (participle-to-noun median 1, 90th percentile 3 against 4). Accuracy falls steadily with distance: UAS is 83.3% for adjacent words and 43.9% for arcs of eight words or more. Long arcs are rarer, but they are where most attachment errors per arc occur.

## Your turn — extension

1. Repeat the gold measurements on the IMST training split. Are the BOUN medians a property of Turkish or of BOUN's genres?
2. Compute the same accuracy-by-distance table for a transition-based parser trained on IMST (for example UDPipe 1 or a Stanza-free arc-standard parser) and compare.